# Covariance with TJPCov

The `CLPCovariance` stage writes the covariance used in the likelihood. It reads
the SACC file written by TXPipe, which has the counts, the stacked
$\Delta\Sigma$ profiles and the covariance of the profiles, and computes the
covariance of the counts with [TJPCov](https://github.com/LSSTDESC/TJPCov). Its
output is the `clusters_sacc_file_cov` file read by `CLPFirecrown`.

Here we compute the covariance of the cosmoDC2 redMaPPer baseline analysis. Its
inputs are in `data/`
- `clusters_sacc_file.sacc`, the SACC file written by TXPipe,
- `fiducial_cosmology.yml`, the cosmoDC2 fiducial cosmology.

Use the `firecrown_clp` kernel, as explained in [README.md](README.md). The stage takes a few minutes.

In [ ]:
import os
import sys
sys.path.append("..")  # not needed if CLPipe is installed with `pip install -e .`

import matplotlib.pyplot as plt
import numpy as np
import sacc

from clpipe.clp_covariance import CLPCovariance

## 1. The input

The TXPipe file has one count per redshift and richness bin, and one
$\Delta\Sigma$ value per radial bin in each of them. Its covariance has the
bootstrap covariance of the profiles, and only the measured counts on the
diagonal of the counts block, as a first estimate of their Poisson variance.

In [ ]:
data = sacc.Sacc.load_fits("data/clusters_sacc_file.sacc")
counts = data.indices("cluster_counts")
profiles = data.indices("cluster_delta_sigma")
print(f"{len(counts)} counts and {len(profiles)} DeltaSigma values")
print("variance of the first counts:", np.diag(data.covariance.covmat)[counts][:4])
print("first counts:", data.mean[counts][:4])

## 2. Configuration

The options of the stage are in the `CLPCovariance` section of the
configuration file. To predict the counts, TJPCov needs the halo mass function
and the mass-richness relation, here the ones of the baseline analysis.
- `cov_type` lists the TJPCov terms, the Gaussian term (shot noise) and the
  super-sample covariance (SSC).
- `photo-z` is the redshift scatter of the clusters, $\sigma_z = \sigma_0(1+z)$.
- `mor_parameters` gives the halo mass function, the halo bias of the SSC term,
  the mass range, and the mass-richness relation at the pivot mass and redshift.

The other options keep their defaults, explained in section 4.

In [ ]:
config = {
    "cov_type": ["ClusterCountsGaussian", "ClusterCountsSSC"],
    "photo-z": {"sigma_0": 0.05},
    "mor_parameters": {
        "mass_func": "Despali16",
        "mass_def": "200c",
        "halo_bias": "Tinker10",
        "min_halo_mass": 1.0e12,
        "max_halo_mass": 3.16e15,
        "m_pivot": 14.3,
        "z_pivot": 0.5,
        "mu_p0": 3.3439,
        "mu_p1": 0.9582,
        "mu_p2": -0.0193,
        "sigma_p0": 0.5623,
        "sigma_p1": 0.0455,
        "sigma_p2": -0.0445,
    },
}

## 3. Running the stage

In [ ]:
os.makedirs("outputs", exist_ok=True)
stage = CLPCovariance.make_stage(
    clusters_sacc_file="data/clusters_sacc_file.sacc",
    fiducial_cosmology="data/fiducial_cosmology.yml",
    clusters_sacc_file_cov="outputs/clusters_sacc_file_cov.sacc",
    **config,
)
stage.run()

Besides `clusters_sacc_file_cov.sacc`, the stage writes the Gaussian and SSC
terms computed by TJPCov in `clusters_sacc_file_cov_gauss.sacc` and
`clusters_sacc_file_cov_SSC.sacc`. TJPCov does not overwrite these files. When
they already exist, it adds the date to the names of the new ones.

## 4. What the stage computed

The counts block is built in two steps. TJPCov computes the Gaussian and SSC
terms with its own prediction of the counts. The stage then replaces the
Gaussian term by the counts predicted by [CROW](https://github.com/LSSTDESC/crow)
with the completeness of the baseline analysis (option `sel_func`), and rescales
the SSC term to these counts (option `replace_tjpcov_cov`). The code marks this
replacement as temporary, until it can be done in TJPCov.

The $\Delta\Sigma$ block is copied from the TXPipe file, keeping only its
diagonal (option `diagonal_shear_covariance`).

In [ ]:
result = sacc.Sacc.load_fits("outputs/clusters_sacc_file_cov.sacc")
cov = result.covariance.covmat
counts = result.indices("cluster_counts")
predicted = np.array([stage.crow_theory_counts[i] for i in counts])
ssc = np.diag(cov)[counts] - predicted
correlation = cov / np.sqrt(np.outer(np.diag(cov), np.diag(cov)))

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.5))
image = ax0.imshow(correlation[np.ix_(counts, counts)], cmap="RdBu_r", vmin=-1, vmax=1)
fig.colorbar(image, ax=ax0, label="correlation")
ax0.set(title="Counts block", xlabel="data point", ylabel="data point")

ax1.plot(np.sqrt(predicted) / predicted, "o-", label="shot noise")
ax1.plot(np.sqrt(ssc) / predicted, "s-", label="super-sample covariance")
ax1.set(yscale="log", xlabel="count index", ylabel="relative error", title="Counts errors")
ax1.legend()
fig.tight_layout()

The counts are ordered by richness bin, then by redshift bin. The shot noise
dominates the error of every count. The SSC term is largest at low redshift,
where the volume of the bins is smallest, and it correlates the counts, mostly
those at the same redshift.

## 5. Comparison with the analysis file

`data/clusters_sacc_file_cov.sacc`, used in the other tutorials, was written by
this stage in the baseline analysis.

In [ ]:
analysis = sacc.Sacc.load_fits("data/clusters_sacc_file_cov.sacc")
print("same data vector:", np.allclose(result.mean, analysis.mean))
print("same covariance:", np.allclose(cov, analysis.covariance.covmat))